# Setup





In [1]:
!nvidia-smi

Wed May 29 16:30:46 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:00:04.0 Off |                    0 |
| N/A   38C    P0              27W / 250W |      0MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!pip install transformers -q
# !pip install pytorch_lightning -q
!pip install datasets -q
!pip install rouge -q
!pip install torch -q
!pip install tqdm -q

In [3]:
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os
import string
import operator
import random

from torch.utils.data import Dataset, DataLoader
from transformers import AdamW
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm, trange

seed = 42
torch.cuda.empty_cache()
device = torch.device('cuda')

# Module Untils

In [4]:
n_gpu = '0'
gradient_accumulation_steps = 1
lr = 1e-4
adam_epsilon = 1e-8
weight_decay = 0.0
num_warmup_steps= 0.0
num_train_epochs = 1
save_model = True
save_last_k = 1
save_last = True
bi_train = False
use_same_model = True
train_batch_size = 8
eval_batch_size = 128
model_checkpoint = 'VietAI/vit5-base' #'google/mt5-base'
max_seq_length = 256
elem_dict = ["subject", "object", "aspect", "predicate", "label"]
data_dir = "/kaggle/input/mtv-mtt-data-aug"

working_dir = "/kaggle/working"
result_dir = f"{working_dir}/result/model"
inference_dir = f"{working_dir}/result/inference"

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
if not os.path.exists(inference_dir):
    os.makedirs(inference_dir)

## Data utils

In [5]:
import re

def read_data_file(data_path):
  with open(data_path, 'r', encoding='UTF-8') as fp:
      sents, labels = [], []
      for line in fp:
          # print(line)
          line = line.rstrip("\n")
          sent, tuples = line.split('===>')
          sents.append(sent)
          # tuples = tuples.replace("'", "")
          labels.append(tuples)

  return sents, labels


In [6]:
def get_max_length(inputs, tokenizer):
    return max(len(tokenizer.encode(i)) for i in inputs)

In [7]:
class MyDataset(Dataset):
    def __init__(self, tokenizer, inputs=None, targets=None):
        self.tokenizer = tokenizer
        self.inputs, self.targets = inputs or [], targets or []
        self.input_tensor_list, self.target_tensor_list = [], []

        self.max_len = max_seq_length

        self.input_tensor_list, self.target_tensor_list = self.encode(self.inputs, self.targets)



    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        source_ids = self.input_tensor_list[idx]["input_ids"].squeeze()
        target_ids = self.target_tensor_list[idx]["input_ids"].squeeze()

        source_mask = self.input_tensor_list[idx]["attention_mask"].squeeze()
        target_mask = self.target_tensor_list[idx]["attention_mask"].squeeze()

        return {"source_ids": source_ids, "source_mask": source_mask, "target_ids": target_ids, "target_mask": target_mask}

    def encode(self, inputs=[], targets=[]):
        input_tensor_list, target_tensor_list = [], []

        for i in range(len(inputs)):
            input_i = ' '.join(inputs[i]) if isinstance(inputs[i], list) else inputs[i]
            target_i = ' '.join(targets[i]) if isinstance(targets[i], list) else targets[i]

            tokenized_input = self.tokenizer.batch_encode_plus([input_i], max_length=self.max_len,padding='max_length', truncation=True, return_tensors="pt")
            tokenized_target = self.tokenizer.batch_encode_plus([target_i], max_length=self.max_len, padding='max_length', truncation=True, return_tensors="pt")

            input_tensor_list.append(tokenized_input)
            target_tensor_list.append(tokenized_target)
            
        return input_tensor_list, target_tensor_list

def get_dataset(file_path, tokenizer, mode="train"):
    if bi_train:
        targets, inputs = read_data_file(file_path)
    else:
        inputs, targets = read_data_file(file_path)
    dataset = MyDataset(tokenizer, inputs=inputs, targets=targets)
    return dataset




In [8]:
# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
# tokenizer.add_tokens(SPECIAL_TOKENS)
        
# # train_inputs, train_labels = read_data_file(os.path.join(data_dir, "small_train.txt"))
# # train_max_length = get_max_length(train_labels, tokenizer)
# # print(tokenizer.tokenize(train_inputs[0], add_special_tokens=True))
# train_data = get_dataset(os.path.join(data_dir, "small_train.txt"), tokenizer=tokenizer)
# print(tokenizer.decode(train_data[0]['source_ids'], skip_special_tokens = False))
# print(tokenizer.decode(train_data[0]['target_ids'], skip_special_tokens = False))

In [9]:
# SPECIAL_TOKENS = ['<sub>', '<obj>', '<asp>', '<pred>', '<lab>', '<unk>', 'COM', 'COM+', 'COM-', 'SUP', 'SUP+', 'SUP-', 'EQL', 'DIF', '(', ')']

# tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
# tokenizer.add_tokens(SPECIAL_TOKENS)
# train_data = get_dataset(os.path.join(data_dir, 'test.txt'), tokenizer)

# max_length = 0
# max_seq = ''
# for data in train_data:
#     if torch.count_nonzero(data['target_ids']) > max_length:
#         max_length = torch.count_nonzero(data['target_ids'])
#         max_seq = tokenizer.convert_ids_to_tokens(data['target_ids'], skip_special_tokens=True) #tokenizer.decode(data['target_ids'])
        
# print(max_seq)

# max_length = 0
# max_seq = ''
# for data in train_data:
#     if torch.count_nonzero(data['source_ids']) > max_length:
#         max_length = torch.count_nonzero(data['source_ids'])
#         max_seq = tokenizer.decode(data['source_ids'], skip_special_tokens=True)
        
# print(max_seq)

## Infer

In [10]:
def calculate_inference_loss(model, tokenizer, input_text, true_sequences):
    # Generate sequences
    with torch.no_grad():
        generated_sequences = model.generate(**inputs)

    # Tokenize true sequences
    true_inputs = tokenizer(true_sequences, return_tensors="pt", truncation=True, padding=True)

    # Forward pass through the model for true sequences
    with torch.no_grad():
        true_outputs = model(**true_inputs)

    # Get logits from the output for true sequences
    true_logits = true_outputs.logits

    # Calculate cross-entropy loss
    loss = torch.nn.functional.cross_entropy(true_logits, generated_sequences.view(-1))

    return loss.item()

In [11]:
SPECIAL_TOKENS = ['[S]', '[O]', '[A]', '[P]', '[L]', '[UNK]', 'COM', 'COM+', 'COM-', 'SUP', 'SUP+', 'SUP-', 'EQL', 'DIF', '(', ')', ';'] # '[A]OSPL]', '[OAPSL]', '[AOSPL]', '[OSPAL]', '[APOSL]', '[OPSAL]', '[PSAOL]', '[POASL]', '[AOPSL]', '[SAPOL]', '[SAOPL]', '[SOAPL]', '[PASOL]', '[PAOSL]', '[SOPAL]', '[SPAOL]', '[OPASL]', '[POSAL]', '[ASPOL]', '[ASOPL]', '[PSOAL]', '[APSOL]', '[OSAPL]', '[SPOAL]', '[PL]', '[APL]', '[PAL]', '[L]']

def prepare_constrained_vocab(name):
    inputs, _ = read_data_file(os.path.join(data_dir, f"{name}.txt"))
    constrained_vocab = set(" ".join(inputs).split())
    constrained_vocab.update(SPECIAL_TOKENS)
    constrained_vocab = list(constrained_vocab)
    return list(SPECIAL_TOKENS)
    
    
    
class Prefix_fn_cls():
    def __init__(self, tokenizer, name, input_enc_idxs):
        self.input_enc_idxs=input_enc_idxs
        self.tokenizer= tokenizer
        self.constrained_vocab = prepare_constrained_vocab(name)
        # only add special_tokens for extract process
        self.special_ids = [element for l in self.tokenizer(self.constrained_vocab, add_special_tokens=False)['input_ids'] for element in l]
        self.special_ids = list(set(self.special_ids))

    def get(self, batch_id, previous_tokens):
        inputs = list(set(self.input_enc_idxs[batch_id].tolist())) + self.special_ids
        return inputs

In [12]:
def infer(dataset, model, tokenizer, batch_size, keep_mask= False, name="eval", constrained=False, **decode_dict):
    data_loader = DataLoader(dataset, batch_size=batch_size, num_workers=4)

    if keep_mask:
        print("Keep mask: ", keep_mask)
        unwanted_tokens = [tokenizer.eos_token, tokenizer.pad_token]
        unwanted_ids = tokenizer.convert_tokens_to_ids(unwanted_tokens)

        def filter_decode(ids):
            ids = [i for i in ids if i not in unwanted_ids]
            tokens = tokenizer.convert_ids_to_tokens(ids)
            sentence = tokenizer.convert_tokens_to_string(tokens)
            return sentence

    inputs, outputs, targets = [], [], []    
    average_loss = 0
    
    model.eval()
    
    if name != "eval":
        with torch.no_grad():
            for batch in tqdm(data_loader, disable=True):
                if constrained:
                    prefix_fn_obj = Prefix_fn_cls(tokenizer, name, batch['source_ids'].to(device))
                    prefix_fn = lambda batch_id, sent: prefix_fn_obj.get(batch_id, sent)
                else:
                    prefix_fn = None
                outs_dict = model.generate(input_ids = batch['source_ids'].to(device),
                                           attention_mask = batch['source_mask'].to(device),
                                           output_scores = True,
                                           return_dict_in_generate = True,
                                           max_length = max_seq_length,
                                           prefix_allowed_tokens_fn = prefix_fn,
                                           **decode_dict)

                outs = outs_dict['sequences']

                if keep_mask:
                    input_ = [filter_decode(ids) for ids in batch['source_ids']]
                    dec = [filter_decode(ids) for ids in outs]
                    target = [filter_decode(ids) for ids in batch['target_ids']]
                else:
                    input_ = [tokenizer.decode(ids, skip_special_tokens=True) for ids in batch['source_ids']]
                    dec = [tokenizer.decode(ids, skip_special_tokens=True) for ids in outs]
                    target = [tokenizer.decode(ids, skip_special_tokens=True) for ids in batch['target_ids']]

                inputs.extend(input_)
                outputs.extend(dec)
                targets.extend(target)
                
    elif name =="eval":
        criterion = nn.CrossEntropyLoss()
        total_loss = 0
        num_batches = len(data_loader)
        with torch.no_grad():
            for batch in tqdm(data_loader, disable=True):
                lm_labels = batch["target_ids"]
                lm_labels[lm_labels[:, :] == tokenizer.pad_token_id] = -100
                outs = model(
                    batch["source_ids"].to(device),
                    attention_mask = batch["source_mask"].to(device),
                    labels = lm_labels.to(device),
                    decoder_attention_mask = batch["target_mask"].to(device),
                    decoder_input_ids = None,
                )

            loss = outs[0]
            total_loss += loss.item()
            
        average_loss = total_loss/num_batches
#         print(f"Average Evaluation Loss: {average_loss}")
                
   
    with open(os.path.join(inference_dir, f"{name}_output_{constrained}.txt"), "w", encoding="utf-8") as f:
        for i, o in enumerate(outputs):
            f.write(f"{inputs[i]} ===> {o}\n")

    
    return average_loss, inputs, outputs, targets




## eval metrics

In [13]:
import copy
from sklearn.metrics import f1_score, precision_recall_fscore_support
import re

def extract_elements(input_string):
    input_list = input_string.split(';')
    pattern = re.compile(r'<sub>(.*?)<obj>(.*?)<asp>(.*?)<pred>(.*?)<lab>(.*?)$')
    result=[]
    for i in input_list:
        i = i.strip()
        match = re.match(pattern, i[1:-1].strip())
        
        if match:
            items = match.groups()  
            new_items = []
            for i in range(len(items)):
                new_items.append(items[i].strip())
            result.append(new_items)
        else:
            result.append(None)
        
    return result

def compute_metrics(predicted_list, gold_list):
    # Transpose the list of tuples to get a list of lists where each list corresponds to a position
    predicted_positions = list(map(list, zip(*predicted_list)))
    gold_positions = list(map(list, zip(*gold_list)))

    precision_scores = []
    recall_scores = []
    micro_f1_scores = []
    macro_f1_scores = []
    f1_scores = []

    # Iterate over each position
    for predicted, gold in zip(predicted_positions, gold_positions):
        # Compute micro-F1 for the position
        micro_f1 = f1_score(predicted, gold, average='micro')
        micro_f1_scores.append(micro_f1)

        # Compute macro-F1 for the position
        macro_f1 = f1_score(predicted, gold, average='macro')
        macro_f1_scores.append(macro_f1)

        # Compute F1-score for the position
        p, r, f1, _ = precision_recall_fscore_support(predicted, gold, average=None)
        f1_scores.append(f1[0])
        precision_scores.append(p[0])
        recall_scores.append(r[0])

    return precision_scores, recall_scores, micro_f1_scores, macro_f1_scores, f1_scores


def eval(pred_tups, gold_tups, verbose="quite", elem_dict=None):
    assert len(pred_tups) == len(gold_tups)

    elem_dict = elem_dict
    all_labels, all_predictions, error_preds = [], [], []
    for index in range(len(gold_tups)):
        predict_list = extract_elements(pred_tups[index])
        gold_list = extract_elements(gold_tups[index])
        
        if len(gold_list) > len(predict_list):
            error_preds.append(f"{index} Incomplete Prediction: {gold_tups[index]} ===> {pred_tups[index]}")
            
        for i in range(len(predict_list)):
            if  i >= len(gold_list):
                error_preds.append(f"{index} Adundant Prediction: {pred_tups[index]}")
            elif predict_list[i] is None or gold_list[i] is None or len(gold_list[i]) != len(predict_list[i]) or len(predict_list[i]) != 5:
                error_preds.append(f"{index}: {gold_tups[index]} ===> {pred_tups[index]}")
            else:
                all_labels.append(gold_list[i])
                all_predictions.append(predict_list[i])
        
        

    precision_scores, recall_scores, micro_f1, macro_f1, f1_scores = compute_metrics(all_predictions, all_labels)

    scores_dict = {}
    for i, elem in enumerate(elem_dict):
        scores_dict[elem] = {"P": precision_scores[i], "R": recall_scores[i], "F1": f1_scores[i], "Marco - F1": macro_f1[i], "Micro - F1": micro_f1[i]}

    with open(os.path.join(inference_dir, 'error_prediction.txt'), 'w', encoding='utf-8') as fout:
        for i, error in enumerate(error_preds):
            fout.write(f"{error}\n")

    print(f"The number of error predictions: {len(error_preds)}")
    if verbose != "quiet":
        print(f"Evaluation Result: {scores_dict}")

    return scores_dict


## train

In [14]:
def train(model, tokenizer, train_data, val_data, epochs, lr, train_batch_size, eval_batch_size, acc_step=None, save_model=False, save_last=False, elem_dict= elem_dict, constrained=False ):
    print("#"*20+" BEGIN TRAINING "+ "#"*20)
    no_decay =["bias", "LayerNorm.Weight"]
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay, },
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0, },
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=lr, eps=adam_epsilon)

    if acc_step is None:
        acc_step = gradient_accumulation_steps
        
    train_loader = DataLoader(train_data, batch_size=train_batch_size, drop_last=True, shuffle=True)
    t_total = (
        (len(train_loader.dataset) // (train_batch_size * max(1, len(n_gpu))))
        // acc_step
        *float(epochs)
    )
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=num_warmup_steps, num_training_steps = t_total)
    train_iterator = trange(int(epochs), dynamic_ncols=True, desc="Epoch")
    
    train_losses, eval_losses = [], []
    for n_epoch, _ in enumerate(train_iterator):
        epoch_train_loss = 0.0
        epoch_iterator = tqdm(train_loader, dynamic_ncols=True, desc="Iteration", disable=True)

        for step, batch in enumerate(epoch_iterator):
            model.train()

            lm_labels = batch["target_ids"]
            lm_labels[lm_labels[:, :] == tokenizer.pad_token_id] = -100
            outputs = model(
                batch["source_ids"].to(device),
                attention_mask = batch["source_mask"].to(device),
                labels = lm_labels.to(device),
                decoder_attention_mask = batch["target_mask"].to(device),
                decoder_input_ids = None,
            )

            loss = outputs[0]
            loss.backward()
            epoch_train_loss += loss.item()

            if (step + 1) % acc_step == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                model.zero_grad()
        
        eval_loss = 0.0
        eval_loss, sents, predictions, golds = infer(val_data, model, tokenizer, batch_size=train_batch_size, name="eval", constrained=constrained)
        eval_losses.append(eval_loss)
        
#         for i in range(5):
#           if i < len(predictions):
#             print(f"{sents[i]} ===> {predictions[i]}")

        # score_dict = eval(predictions, golds, verbose="info", elem_dict= elem_dict)

#         if save_model and n_epoch in range(num_train_epochs)[-save_last_k:]:
#             save_dir = os.path.join(result_dir, f"{model_checkpoint}_checkpoint-e{n_epoch}-constrained-{constrained}")
#             if not os.path.exists(save_dir):
#                 os.makedirs(save_dir)

#             model.save_pretrained(save_dir)
#             tokenizer.save_pretrained(save_dir)

#             print(f"Save model checkpoint to {save_dir}")
        
        train_losses.append(epoch_train_loss / len(epoch_iterator))
        
        print(f"Epoch {n_epoch} - Average epoch train loss: {epoch_train_loss / len(epoch_iterator):.5f} lr: {scheduler.get_last_lr()}")
        print(f"Average Evaluation Loss: {eval_loss:.5f}")

    if save_last:
        save_dir = os.path.join(result_dir, f"{model_checkpoint.split('/')[-1]}-e{n_epoch}-extract-tuple-constrained-model-{constrained}")
        if not os.path.exists(save_dir):
            os.makedirs(save_dir)

        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)

        print(f"Save model checkpoint to {save_dir}")
    
    with open(os.path.join(inference_dir, "train_losses.txt"), 'w') as f:
        for loss in train_losses:
            f.write(f"{loss}\n")
    
    with open(os.path.join(inference_dir, "eval_losses.txt"), 'w') as f:
        for loss in eval_losses:
            f.write(f"{loss}\n")

    print("#"*20+" FINISH TRAINING "+ "#"*20)


# Run

In [15]:
# do_train = True
# do_test = True
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def main(do_train, do_test, test_label, constrained=False):
    if do_train:
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        tokenizer.add_tokens(SPECIAL_TOKENS)
        
        train_inputs, train_labels = read_data_file(os.path.join(data_dir, "train.txt"))
        train_max_length = get_max_length(train_labels, tokenizer)
        eval_inputs, eval_labels = read_data_file(os.path.join(data_dir, "dev.txt"))
        eval_max_length = get_max_length(eval_labels, tokenizer)
        eval_max_length = 0

        train_data = get_dataset(os.path.join(data_dir, "train.txt"), tokenizer=tokenizer)
        eval_data = get_dataset(os.path.join(data_dir, "dev.txt"), tokenizer=tokenizer)
        
        for i in range(3):
            print(tokenizer.decode(train_data[i]['source_ids'], skip_special_tokens = True))
            print(tokenizer.decode(train_data[i]['target_ids'], skip_special_tokens = True))

        model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
        model.resize_token_embeddings(len(tokenizer))
        model.to(device)

        print("*"*20+" Training "+"*"*20)
        print(f"Train: {len(train_data)}, Model: {model_checkpoint}")
        print(f"Train Label Max Length : {train_max_length}\n")
        
        train(model, tokenizer, train_data, val_data=eval_data, epochs=num_train_epochs, lr=lr, train_batch_size=train_batch_size, eval_batch_size=eval_batch_size, save_model=True, save_last=True, elem_dict=elem_dict, constrained=constrained)

    
#     if bi_train:
#         if not use_same_model:
#             tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
#             tokenizer.add_tokens(SPECIAL_TOKENS)
#             model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
#             model.resize_token_embeddings(len(tokenizer))
#             model.to(device)
        
#         train_labels, train_inputs = read_data_file(os.path.join(data_dir, "train.txt"))
#         train_max_length = get_max_length(train_labels, tokenizer)
#         eval_labels, eval_inputs = read_data_file(os.path.join(data_dir, "dev.txt"))
#         eval_max_length = get_max_length(eval_labels, tokenizer)
        
#         print("*"*20+"Bidirectional Training "+"*"*20)
#         print(f"Train: {len(train_data)}, Eval: {len(eval_data)}, Model: {model_checkpoint}")
#         print(f"Train Label Max Length : {train_max_length}\nEval Label Max Length: {eval_max_length}")
        
#         train(model, tokenizer, train_data, eval_data, epochs=num_train_epochs, lr=lr, train_batch_size=train_batch_size, eval_batch_size=eval_batch_size, save_model=True, save_last=True, elem_dict=elem_dict, constrained=constrained)
#         _, targets, gen_sents, _ = infer(test_data, model, tokenizer, batch_size=train_batch_size, name="test", constrained=constrained)
        
#         aug_sents = train_inputs + gen_sents
#         aug_labels = train_labels + targets
#         assert len(aug_sents) == len(aug_labels)
        
#         with open(r'kaggle/aug_train.txt', 'w') as fp:
#             for i, sent in enumerate(aug_sents):
#                 fp.write("%s\n" % f'{sent}===>{aug_labels[i]}')

    if do_test:
        print("*"*20+" TESTING "+"*"*20)
        all_checkpoints = []
        saved_model_dir = result_dir

        for f in os.listdir(saved_model_dir):
            file_name = os.path.join(saved_model_dir, f)
            if 'constrained-model' in f and model_checkpoint.split('/')[-1] in f:
                all_checkpoints.append(file_name)
        
    
        test_inputs, _ = read_data_file(os.path.join(data_dir, "test.txt"))
        print(f"Test: {len(test_inputs)}")

        best_f1, best_checkpoint, best_epoch = -999999.0, None, None
        best_score_dict, best_pred_dict = None, None
        all_epochs = []

    #     del model
#         print("*"*20+" Testing "+"*"*20)

        for checkpoint in all_checkpoints:
            model_name = checkpoint.split('/')[-1]
#             epoch = checkpoint.split('-')[-1][1:]
#             all_epochs.append(epoch)
            print(f"Load model from checkpoint {checkpoint}")

            model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
            tokenizer = AutoTokenizer.from_pretrained(checkpoint)
              
            test_data = get_dataset(os.path.join(data_dir, "test.txt"), tokenizer=tokenizer)
              
            model.to(device)

            _, sents, predictions, golds = infer(test_data, model, tokenizer, batch_size=train_batch_size, name="test", constrained=constrained)

            for i in range(5):
              if i < len(predictions):
                print(f"{sents[i]} ===> {predictions[i]}")
              
            if test_label:
                score_dict = eval(predictions, golds, verbose="info", elem_dict=elem_dict)

            with open(f"{inference_dir}/test-{model_name}.txt", 'w', encoding="utf-8") as fout:
                for i, s in enumerate(sents):
                    fout.write(f"{s} ===> {predictions[i]}\n")


# TEST

In [16]:
data_dir = "/kaggle/input/test-sample-4"
main(do_train = False, do_test = True, test_label = False, constrained=True)

******************** TESTING ********************
Test: 46
Load model from checkpoint /kaggle/working/result/model/vit5-base-e0-extract-tuple-constrained-model-True


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


iPhone 14 nhanh hơn Samsung nhưng giá đắt hơn. [S][O][A][P][L] ===> ([S] iPhone 14 [O] Samsung [A] [UNK] [P] nhanh hơn [L] COM+);([S] iPhone 14 [O] Samsung [A] giá [P] đắt hơn [L] COM-)
Ngọn núi Everest cao nhất thế giới. [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])
Hạt bụi nhỏ bé hơn nhiều so với hạt cát. [S][O][A][P][L] ===> ([S] Hạt bụi [O] hạt cát [A] [UNK] [P] nhỏ bé hơn nhiều [L] COM-)
Hút thuốc lá nguy hiểm hơn uống rượu bia. [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] Hút thuốc lá [P] nguy hiểm hơn [L] COM+)
Sống trong khu vực ô nhiễm môi trường nguy hiểm hơn sống trong môi trường sạch. [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])


# TRAIN

In [17]:
main(do_train = True, do_test = True, test_label = False, constrained=True)

tokenizer_config.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

Tuy chênh lệch không nhiều, nhưng rõ ràng việc Galaxy S22 Ultra nổi bật hơn về thông số là điều khó có thể phủ nhận! [P][A][O][S][L]
([P] nổi bật hơn [A] thông số [O] [UNK] [S] Galaxy S22 Ultra [L] COM+)
Galaxy A12 với thiết kế mặt lưng được chia làm 2 phần với phần trên được làm dạng nhám sần, phần dưới là dạng nhám trơn tạo nên điểm khác biệt so với những Smartphone giá rẻ khác. [P][S][O][A][L]
([P] khác biệt [S] Galaxy A12 [O] những Smartphone giá rẻ khác [A] thiết kế mặt lưng [L] DIF)
Galaxy A32 cũng không hề kém cạnh khi sở hữu ống kính có độ phân giải lớn hơn 64 MP và camera phụ lần lượt là 8MP, 5MP và 5MP. [A][P][L]
([A] ống kính [P] không hề kém cạnh [L] EQL)


...

Epoch: 100%|██████████| 1/1 [44:38<00:00, 2678.99s/it]

Epoch 0 - Average epoch train loss: 0.08810 lr: [0.0, 0.0]
Average Evaluation Loss: 0.00076


Save model checkpoint to /kaggle/working/result/model/vit5-base-e0-extract-tuple-constrained-model-True
#################### FINISH TRAINING ####################
******************** TESTING ********************
Test: 3270
Load model from checkpoint /kaggle/working/result/model/vit5-base-e0-extract-tuple-constrained-model-True


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Title : Góc thắc mắc : Nên mua iPhone 14 hay Samsung Galaxy S22? [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])
Hiện nay, iPhone 14 và Samsung Galaxy S22 là hai cái tên rất  hot , liên tục khuấy đảo thị trường điện thoại phân khúc cao cấp. [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])
Đứng trước sự lựa chọn giữa hai  siêu phẩm  đến từ hai thương hiệu đình đám Apple và Samsung, nhiều người vẫn rất phân vân không biết nên lựa chọn sản phẩm nào thì tốt hơn. [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])
Để đưa ra quyết định chọn mua đúng đắn, mời bạn tham khảo bài viết của Nguyễn Kim nhé! [S][O][A][P][L] ===> ([S] [UNK] [O] [UNK] [A] [UNK] [P] [UNK] [L] [UNK])
Tháng 9 vừa qua, các tín đồ yêu công nghệ tỏ ra vô cùng thích thú vì sau thời gian dài chờ đợi thì Apple cũng đã chính thức trình làng các sản phẩm trong iPhone 14 Series với những cải tiến ấn tượng về thiết kế lẫn hiệu năng. [S][O][A][P][L] ===> ([S] [UN

In [13]:
data_dir = "/kaggle/input/test-sample-1"
main(do_train = False, do_test = True, test_label = False, constrained=True)

******************** TESTING ********************
Test: 1
Load model from checkpoint /kaggle/working/result/model/vit5-base-e0-extract-tuple-constrained-model-True


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


iPhone 14 nhanh hơn Samsung nhưng giá đắt hơn [S][O][A][P][L] ===> ([S] iPhone 14 [O] Samsung [A] [UNK] [P] nhanh hơn [L] COM+);([S] iPhone 14 [O] Samsung [A] giá [P] đắt hơn [L] COM-)


## Infer augmented data

In [18]:
# # do_train = True
# # do_test = True
# import os
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# def main(do_train, do_test, test_label, constrained=False):
#     if do_train:
#         tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
#         tokenizer.add_tokens(SPECIAL_TOKENS)
        
#         train_inputs, train_labels = read_data_file(os.path.join(data_dir, "train.txt"))
#         train_max_length = get_max_length(train_labels, tokenizer)
#         eval_inputs, eval_labels = read_data_file(os.path.join(data_dir, "dev.txt"))
#         eval_max_length = get_max_length(eval_labels, tokenizer)
#         eval_max_length = 0

#         train_data = get_dataset(os.path.join(data_dir, "train.txt"), tokenizer=tokenizer)
#         eval_data = get_dataset(os.path.join(data_dir, "dev.txt"), tokenizer=tokenizer)
        
#         for i in range(3):
#             print(tokenizer.decode(train_data[i]['source_ids'], skip_special_tokens = True))
#             print(tokenizer.decode(train_data[i]['target_ids'], skip_special_tokens = True))

#         model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
#         model.resize_token_embeddings(len(tokenizer))
#         model.to(device)

#         print("*"*20+" Training "+"*"*20)
#         print(f"Train: {len(train_data)}, Model: {model_checkpoint}")
#         print(f"Train Label Max Length : {train_max_length}\n")
        
#         train(model, tokenizer, train_data, val_data=eval_data, epochs=num_train_epochs, lr=lr, train_batch_size=train_batch_size, eval_batch_size=eval_batch_size, save_model=True, save_last=True, elem_dict=elem_dict, constrained=constrained)

    
# #     if bi_train:
# #         if not use_same_model:
# #             tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
# #             tokenizer.add_tokens(SPECIAL_TOKENS)
# #             model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
# #             model.resize_token_embeddings(len(tokenizer))
# #             model.to(device)
        
# #         train_labels, train_inputs = read_data_file(os.path.join(data_dir, "train.txt"))
# #         train_max_length = get_max_length(train_labels, tokenizer)
# #         eval_labels, eval_inputs = read_data_file(os.path.join(data_dir, "dev.txt"))
# #         eval_max_length = get_max_length(eval_labels, tokenizer)
        
# #         print("*"*20+"Bidirectional Training "+"*"*20)
# #         print(f"Train: {len(train_data)}, Eval: {len(eval_data)}, Model: {model_checkpoint}")
# #         print(f"Train Label Max Length : {train_max_length}\nEval Label Max Length: {eval_max_length}")
        
# #         train(model, tokenizer, train_data, eval_data, epochs=num_train_epochs, lr=lr, train_batch_size=train_batch_size, eval_batch_size=eval_batch_size, save_model=True, save_last=True, elem_dict=elem_dict, constrained=constrained)
# #         _, targets, gen_sents, _ = infer(test_data, model, tokenizer, batch_size=train_batch_size, name="test", constrained=constrained)
        
# #         aug_sents = train_inputs + gen_sents
# #         aug_labels = train_labels + targets
# #         assert len(aug_sents) == len(aug_labels)
        
# #         with open(r'kaggle/aug_train.txt', 'w') as fp:
# #             for i, sent in enumerate(aug_sents):
# #                 fp.write("%s\n" % f'{sent}===>{aug_labels[i]}')

#     if do_test:
#         print("*"*20+" TESTING "+"*"*20)
#         all_checkpoints = []
#         saved_model_dir = result_dir

#         for f in os.listdir(saved_model_dir):
#             file_name = os.path.join(saved_model_dir, f)
#             if 'constrained-model' in f and model_checkpoint.split('/')[-1] in f:
#                 all_checkpoints.append(file_name)
        
    
# #         test_inputs, _ = read_data_file(os.path.join(data_dir, "test.txt"))
# #         print(f"Test: {len(test_inputs)}")

#         best_f1, best_checkpoint, best_epoch = -999999.0, None, None
#         best_score_dict, best_pred_dict = None, None
#         all_epochs = []

#     #     del model
# #         print("*"*20+" Testing "+"*"*20)

#         for checkpoint in all_checkpoints:
#             model_name = checkpoint.split('/')[-1]
# #             epoch = checkpoint.split('-')[-1][1:]
# #             all_epochs.append(epoch)
#             print(f"Load model from checkpoint {checkpoint}")

#             model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
#             tokenizer = AutoTokenizer.from_pretrained(checkpoint)
              
# #             test_data = get_dataset(os.path.join(data_dir, "test.txt"), tokenizer=tokenizer)
#             test_data = get_dataset('/kaggle/input/aug-sents/augmented_sent.txt', tokenizer)
              
#             model.to(device)

#             _, sents, predictions, golds = infer(test_data, model, tokenizer, batch_size=train_batch_size, name="test", constrained=constrained)

#             for i in range(5):
#               if i < len(predictions):
#                 print(f"{sents[i]} ===> {predictions[i]}")
              
#             if test_label:
#                 score_dict = eval(predictions, golds, verbose="info", elem_dict=elem_dict)

#             with open(f"{inference_dir}/aug-{model_name}.txt", 'w', encoding="utf-8") as fout:
#                 for i, s in enumerate(sents):
#                     fout.write(f"{s} ===> {predictions[i]}\n")


In [19]:
# main(do_train = False, do_test = True, test_label = False, constrained=True)

In [20]:
# !rm -rf /kaggle/working/* 

In [21]:
# !zip -r model_mtv_mtt.zip /kaggle/working/result/model/vit5-base-e0-extract-tuple-constrained-model-True

## XAI for COQE (T5 Seq2Seq)

Ap dung cac ky thuat giai thich theo phong cach KAGGLE_FULL_PIPELINE, nhung dieu chinh cho bai toan sinh chuoi quintuple (subject, object, aspect, predicate, label).

In [ ]:
# XAI setup + load model/checkpoint
!pip install -q captum shap lime seaborn

import os
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from lime.lime_text import LimeTextExplainer
from captum.attr import LayerIntegratedGradients, visualization
import shap

sns.set_theme(style='whitegrid')

if 'device' not in globals():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

XAI_OUT_DIR = '/kaggle/working/coqe_xai_outputs'
os.makedirs(XAI_OUT_DIR, exist_ok=True)


def find_latest_checkpoint(base_dir, model_name='vit5-base'):
    if not os.path.exists(base_dir):
        return None
    cands = []
    for p in glob.glob(os.path.join(base_dir, '*constrained-model-*')):
        bn = os.path.basename(p)
        if model_name in bn:
            cands.append(p)
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: os.path.getmtime(p), reverse=True)
    return cands[0]


if 'model' in globals() and 'tokenizer' in globals():
    print('Reuse existing model/tokenizer from current kernel.')
    COQE_MODEL = model
    COQE_TOKENIZER = tokenizer
else:
    model_name_tag = model_checkpoint.split('/')[-1] if 'model_checkpoint' in globals() else 'vit5-base'
    ckpt = find_latest_checkpoint(result_dir if 'result_dir' in globals() else '/kaggle/working/result/model', model_name_tag)
    if ckpt is None:
        raise FileNotFoundError('Khong tim thay checkpoint da train. Hay train truoc hoac chi ro checkpoint.')
    print('Load checkpoint:', ckpt)
    COQE_TOKENIZER = AutoTokenizer.from_pretrained(ckpt)
    COQE_MODEL = AutoModelForSeq2SeqLM.from_pretrained(ckpt)

COQE_MODEL.to(device)
COQE_MODEL.eval()


def generate_coqe(text, max_new_tokens=64, num_beams=4):
    enc = COQE_TOKENIZER(text, return_tensors='pt', truncation=True, max_length=max_seq_length).to(device)
    with torch.no_grad():
        out = COQE_MODEL.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
        )
    return COQE_TOKENIZER.decode(out[0], skip_special_tokens=True)


if 'EXPLAIN_TEXT' not in globals():
    if 'data_dir' in globals() and os.path.exists(os.path.join(data_dir, 'test.txt')):
        _sents, _ = read_data_file(os.path.join(data_dir, 'test.txt'))
        EXPLAIN_TEXT = _sents[0] if len(_sents) else 'The phone is better than the tablet in battery life.'
    else:
        EXPLAIN_TEXT = 'The phone is better than the tablet in battery life.'

PRED_SEQ = generate_coqe(EXPLAIN_TEXT)
print('EXPLAIN_TEXT:', EXPLAIN_TEXT)
print('PRED_SEQ    :', PRED_SEQ)
print('XAI_OUT_DIR :', XAI_OUT_DIR)

### 1) LIME-style Token Importance (Seq2Seq confidence)

In [ ]:
def seq2seq_confidence(input_text, target_text=None):
    # Confidence proxy cho seq2seq: exp(-teacher-forcing loss)
    if target_text is None:
        target_text = generate_coqe(input_text)

    enc = COQE_TOKENIZER(input_text, return_tensors='pt', truncation=True, max_length=max_seq_length).to(device)
    tgt = COQE_TOKENIZER(target_text, return_tensors='pt', truncation=True, max_length=max_seq_length).input_ids.to(device)

    labels = tgt.clone()
    labels[labels == COQE_TOKENIZER.pad_token_id] = -100

    with torch.no_grad():
        out = COQE_MODEL(**enc, labels=labels)
        loss = out.loss
    conf = float(torch.exp(-loss).item())
    return conf


def lime_predict_fn(texts):
    vals = []
    for t in texts:
        c = seq2seq_confidence(t, PRED_SEQ)
        c = float(np.clip(c, 1e-6, 1 - 1e-6))
        vals.append([1.0 - c, c])
    return np.array(vals)


explainer = LimeTextExplainer(class_names=['low_confidence', 'high_confidence'])
lime_exp = explainer.explain_instance(
    EXPLAIN_TEXT,
    lime_predict_fn,
    num_features=12,
    num_samples=1000,
)

print('Predicted sequence:', PRED_SEQ)
print('Confidence        :', round(seq2seq_confidence(EXPLAIN_TEXT, PRED_SEQ), 4))
print('\nTop token contributions (toward high_confidence):')
for w, s in lime_exp.as_list(label=1):
    print(f'{w}: {s:.4f}')

lime_html = os.path.join(XAI_OUT_DIR, 'lime_seq2seq.html')
with open(lime_html, 'w', encoding='utf-8') as f:
    f.write(lime_exp.as_html())
print('Saved:', lime_html)

lime_exp.show_in_notebook(text=True)

### 2) Integrated Gradients (Encoder tokens for first decoded token)

In [ ]:
enc = COQE_TOKENIZER(
    EXPLAIN_TEXT,
    return_tensors='pt',
    truncation=True,
    max_length=max_seq_length,
).to(device)

with torch.no_grad():
    gen_ids = COQE_MODEL.generate(**enc, max_new_tokens=64, num_beams=4, early_stopping=True)

# Dung token giai ma dau tien (sau decoder start token) lam muc tieu attribution
if gen_ids.shape[1] > 1:
    target_token_id = int(gen_ids[0, 1].item())
else:
    target_token_id = int(gen_ids[0, 0].item())

decoder_start_id = COQE_MODEL.config.decoder_start_token_id
if decoder_start_id is None:
    decoder_start_id = COQE_TOKENIZER.pad_token_id

decoder_input_ids = torch.tensor([[decoder_start_id]], device=device)


def t5_forward_for_ig(input_ids, attention_mask):
    out = COQE_MODEL(
        input_ids=input_ids,
        attention_mask=attention_mask,
        decoder_input_ids=decoder_input_ids,
        return_dict=True,
    )
    return out.logits[:, -1, target_token_id]


lig = LayerIntegratedGradients(t5_forward_for_ig, COQE_MODEL.get_input_embeddings())
baseline_ids = torch.full_like(enc['input_ids'], COQE_TOKENIZER.pad_token_id)

attributions, delta = lig.attribute(
    inputs=enc['input_ids'],
    baselines=baseline_ids,
    additional_forward_args=(enc['attention_mask'],),
    return_convergence_delta=True,
)

attr_sum = attributions.sum(dim=-1).squeeze(0)
norm = torch.norm(attr_sum)
if norm.item() == 0:
    attr_norm = attr_sum
else:
    attr_norm = attr_sum / norm

tokens = COQE_TOKENIZER.convert_ids_to_tokens(enc['input_ids'][0].detach().cpu().tolist())
target_token = COQE_TOKENIZER.convert_ids_to_tokens([target_token_id])[0]

print('Predicted sequence:', PRED_SEQ)
print('Target decoded token for IG:', target_token)

visualization.visualize_text([
    visualization.VisualizationDataRecord(
        attr_norm.detach().cpu(),
        seq2seq_confidence(EXPLAIN_TEXT, PRED_SEQ),
        PRED_SEQ,
        'n/a',
        PRED_SEQ,
        float(attr_norm.sum().item()),
        tokens,
        delta.detach().cpu(),
    )
])

ig_scores = attr_norm.detach().cpu().numpy()
ig_colors = ['#d62728' if s > 0 else '#1f77b4' for s in ig_scores]
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(tokens)), ig_scores, color=ig_colors)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=90)
ax.set_ylabel('Normalized attribution')
ax.set_title('Integrated Gradients over encoder tokens (T5)')
plt.tight_layout()
ig_path = os.path.join(XAI_OUT_DIR, 'integrated_gradients_t5.png')
plt.savefig(ig_path, dpi=150)
plt.show()
print('Saved:', ig_path)

### 3) SHAP Text Explanation (Seq2Seq confidence)

In [ ]:
def shap_predict_fn(texts):
    return np.array([seq2seq_confidence(t, PRED_SEQ) for t in texts])

# Dung masker theo whitespace de on dinh voi tokenizer SentencePiece
masker = shap.maskers.Text(r'\s+')
explainer = shap.Explainer(shap_predict_fn, masker)

shap_values = explainer([EXPLAIN_TEXT])

print('Predicted sequence:', PRED_SEQ)
print('Confidence        :', round(seq2seq_confidence(EXPLAIN_TEXT, PRED_SEQ), 4))

shap.initjs()
shap.plots.text(shap_values[0])

### 4) Attention Weights (Encoder last layer, mean heads)

In [ ]:
enc = COQE_TOKENIZER(
    EXPLAIN_TEXT,
    return_tensors='pt',
    truncation=True,
    max_length=max_seq_length,
).to(device)

decoder_start_id = COQE_MODEL.config.decoder_start_token_id
if decoder_start_id is None:
    decoder_start_id = COQE_TOKENIZER.pad_token_id

decoder_input_ids = torch.tensor([[decoder_start_id]], device=device)

with torch.no_grad():
    out_att = COQE_MODEL(
        input_ids=enc['input_ids'],
        attention_mask=enc['attention_mask'],
        decoder_input_ids=decoder_input_ids,
        output_attentions=True,
        return_dict=True,
    )

enc_att = out_att.encoder_attentions[-1].mean(dim=1).squeeze(0).detach().cpu().numpy()
att_tokens = COQE_TOKENIZER.convert_ids_to_tokens(enc['input_ids'][0].detach().cpu().tolist())

plt.figure(figsize=(12, 10))
plt.imshow(enc_att, cmap='coolwarm', interpolation='nearest')
plt.xticks(ticks=range(len(att_tokens)), labels=att_tokens, rotation=45)
plt.yticks(ticks=range(len(att_tokens)), labels=att_tokens)

for i in range(len(att_tokens)):
    for j in range(len(att_tokens)):
        plt.text(j, i, f'{enc_att[i, j]:.2f}', ha='center', va='center', color='white', fontsize=5)

plt.xlabel('Source Tokens')
plt.ylabel('Target Tokens')
plt.title('T5 Encoder Last-layer Attention (mean over heads)')
plt.colorbar(label='Attention Weight')
plt.tight_layout()
att_path = os.path.join(XAI_OUT_DIR, 'attention_heatmap_t5.png')
plt.savefig(att_path, dpi=150)
plt.show()
print('Saved:', att_path)

### 5) Gradient-based Token Saliency (Grad x Input on encoder embeddings)

In [ ]:
enc = COQE_TOKENIZER(
    EXPLAIN_TEXT,
    return_tensors='pt',
    truncation=True,
    max_length=max_seq_length,
).to(device)

with torch.no_grad():
    gen_ids = COQE_MODEL.generate(**enc, max_new_tokens=64, num_beams=4, early_stopping=True)
if gen_ids.shape[1] > 1:
    target_token_id = int(gen_ids[0, 1].item())
else:
    target_token_id = int(gen_ids[0, 0].item())

decoder_start_id = COQE_MODEL.config.decoder_start_token_id
if decoder_start_id is None:
    decoder_start_id = COQE_TOKENIZER.pad_token_id

decoder_input_ids = torch.tensor([[decoder_start_id]], device=device)

COQE_MODEL.zero_grad()
embed_layer = COQE_MODEL.get_input_embeddings()
inputs_embeds = embed_layer(enc['input_ids']).detach().requires_grad_(True)

out = COQE_MODEL(
    inputs_embeds=inputs_embeds,
    attention_mask=enc['attention_mask'],
    decoder_input_ids=decoder_input_ids,
    return_dict=True,
)

score = out.logits[:, -1, target_token_id].sum()
score.backward()

grads = inputs_embeds.grad
saliency = (grads * inputs_embeds).sum(dim=-1).squeeze(0)
abs_saliency = saliency.abs()
if torch.max(abs_saliency) > 0:
    abs_saliency = abs_saliency / torch.max(abs_saliency)

grad_scores = abs_saliency.detach().cpu().numpy()
grad_tokens = COQE_TOKENIZER.convert_ids_to_tokens(enc['input_ids'][0].detach().cpu().tolist())

print('Predicted sequence:', PRED_SEQ)
print('Gradient target token:', COQE_TOKENIZER.convert_ids_to_tokens([target_token_id])[0])
print('Top token saliency:')
for tok, sc in sorted(list(zip(grad_tokens, grad_scores)), key=lambda x: x[1], reverse=True)[:15]:
    print(f'{tok}: {sc:.4f}')

plt.figure(figsize=(14, 4))
plt.bar(range(len(grad_tokens)), grad_scores, color=['#F58518' if s > 0.5 else '#4C78A8' for s in grad_scores])
plt.xticks(range(len(grad_tokens)), grad_tokens, rotation=90)
plt.ylabel('Normalized saliency')
plt.title('Grad x Input Token Saliency (T5 encoder)')
plt.tight_layout()
grad_path = os.path.join(XAI_OUT_DIR, 'gradxinput_t5.png')
plt.savefig(grad_path, dpi=150)
plt.show()
print('Saved:', grad_path)

### 6) XAI Summary + Export

In [ ]:
import json
import shutil
from datetime import datetime

summary = {
    'model_checkpoint': COQE_MODEL.config._name_or_path if hasattr(COQE_MODEL.config, '_name_or_path') else 'unknown',
    'device': str(device),
    'explain_text': EXPLAIN_TEXT,
    'predicted_quintuple_sequence': PRED_SEQ,
    'xai_outputs': {
        'lime_html': os.path.join(XAI_OUT_DIR, 'lime_seq2seq.html'),
        'integrated_gradients': os.path.join(XAI_OUT_DIR, 'integrated_gradients_t5.png'),
        'attention_heatmap': os.path.join(XAI_OUT_DIR, 'attention_heatmap_t5.png'),
        'gradient_saliency': os.path.join(XAI_OUT_DIR, 'gradxinput_t5.png'),
    },
    'created_at': datetime.utcnow().isoformat() + 'Z',
}

summary_path = os.path.join(XAI_OUT_DIR, 'coqe_xai_summary.json')
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive('/kaggle/working/coqe_xai_bundle', 'zip', XAI_OUT_DIR)

print('=== COQE XAI SUMMARY ===')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Saved summary:', summary_path)
print('Saved zip    :', zip_path)